# Assignment - Sentiment Analysis on Tweet

Step 1 : Import Required Libraries

In [ ]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Baseline model
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import pickle

# BERT
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments

Step 2 : Load Dataset

In [ ]:
# Load the dataset tweet.csv
df = pd.read_csv("/content/tweets.csv")

In [ ]:
# Use only required columns by removing the unwanted 'id' column
df = df[['label', 'tweet']]

In [ ]:
# Display the first few records
df.head()

,label,tweet
0,0,#fingerprint #Pregnancy Test https://goo.gl/h1...
1,0,Finally a transparant silicon case ^^ Thanks t...
2,0,We love this! Would you go? #talk #makememorie...
3,0,I'm wired I know I'm George I was made that wa...
4,1,What amazing service! Apple won't even talk to...


In [ ]:
# Check dataset info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7920 entries, 0 to 7919
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   7920 non-null   int64 
 1   tweet   7920 non-null   object
dtypes: int64(1), object(1)
memory usage: 123.9+ KB


Step 3 : Data Preprocessing

In [ ]:
# Defining a function to clean tweets which contains URLS, hashtags, @ symbols, punctuations etc.
def clean_tweets(tweet):
    tweet = tweet.lower()  # Convert to lowercase

    tweet = re.sub(r"http\S+", "", tweet)  # This is to remove URLs from the tweets
    tweet = re.sub(r"@\w+", "", tweet)     # This is to remove the mentions like @names or ids
    tweet = re.sub(r"#", "", tweet)        # This is to remove the hashtag symbols

    tweet = re.sub(r"[^\w\s]", "", tweet)  # This is for removing the punctuations
    return tweet

In [ ]:
# Apply cleaning
df['tweet'] = df['tweet'].apply(clean_tweets)

In [ ]:
# Display first few records of cleaned tweet column
df[['label','tweet']].head()

,label,tweet
0,0,fingerprint pregnancy test android apps beaut...
1,0,finally a transparant silicon case thanks to ...
2,0,we love this would you go talk makememories un...
3,0,im wired i know im george i was made that way ...
4,1,what amazing service apple wont even talk to m...


Step 4 : Train-Test Split

In [ ]:
# Split the dataset to X and y
X = df['tweet']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# MODEL 1: TF-IDF + Logistic Regression

Step 5 : Feature Extraction

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

Step 6 : Train Model

In [ ]:
model_lr = LogisticRegression()
model_lr.fit(X_train_tfidf, y_train)

LogisticRegression()

Step 7 : Evaluate Model

In [ ]:
y_pred_lr = model_lr.predict(X_test_tfidf)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
print("Logistic Regression Accuracy:", accuracy_lr)

Logistic Regression Accuracy: 0.8630050505050505


Step 8 : Save the Model

In [ ]:
pickle.dump(model_lr, open("logistic_model.pkl", "wb"))
pickle.dump(vectorizer, open("tfidf_vectorizer.pkl", "wb"))

# MODEL 2: BERT (Advanced Model)

Step 9 : Prepare Data

In [ ]:
train_texts = X_train.tolist()
test_texts = X_test.tolist()

train_labels = y_train.tolist()
test_labels = y_test.tolist()

Step 10 : Tokenization

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(train_texts, truncation=True, padding=True)
test_encodings = tokenizer(test_texts, truncation=True, padding=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Step 11 : Dataset Class

In [ ]:
class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TweetDataset(train_encodings, train_labels)
test_dataset = TweetDataset(test_encodings, test_labels)

Step 12 : Load BERT Model

In [ ]:
model_bert = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step 13 : Training Setup

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    logging_steps=10
)

Step 14 : Train Model

In [ ]:
trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.633109
20,0.523944
30,0.344240
40,0.222211
50,0.439800
60,0.204563


Step,Training Loss
10,0.633109
20,0.523944
30,0.344240
40,0.222211
50,0.439800
60,0.204563
70,0.414183
80,0.306778
90,0.256200
100,0.226079


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=792, training_loss=0.3018158586564088, metrics={'train_runtime': 3767.8129, 'train_samples_per_second': 1.682, 'train_steps_per_second': 0.21, 'total_flos': 257223945496320.0, 'train_loss': 0.3018158586564088, 'epoch': 1.0})

Step 15 : Evaluate BERT

In [ ]:
predictions = trainer.predict(test_dataset)

y_pred_bert = predictions.predictions.argmax(axis=1)

accuracy_bert = accuracy_score(test_labels, y_pred_bert)
print("BERT Accuracy:", accuracy_bert)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


BERT Accuracy: 0.9103535353535354


Step 16 : Save BERT Model

In [ ]:
model_bert.save_pretrained("bert_model")
tokenizer.save_pretrained("bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_model/tokenizer_config.json', 'bert_model/tokenizer.json')

FINAL COMPARISON

In [ ]:
print("\nFinal Comparison:")
print("Logistic Regression Accuracy:", accuracy_lr)
print("BERT Accuracy:", accuracy_bert)


Final Comparison:
Logistic Regression Accuracy: 0.8630050505050505
BERT Accuracy: 0.9103535353535354


Step 17 : Load Saved BERT Model

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# Load saved model
model = BertForSequenceClassification.from_pretrained("bert_model")
tokenizer = BertTokenizer.from_pretrained("bert_model")

model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

Step 18 : Create Prediction Function

In [ ]:
def predict_sentiment(tweet):
    tweet = clean_tweets(tweet)

    inputs = tokenizer(tweet, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()

    return "Positive" if pred == 0 else "Negative"

Step 19 : Test with Sample Data

In [ ]:
test_tweets = [
    "I love this new phone, battery life is good",
    "This laptop is very slow and performance is bad"
]

for tweet in test_tweets:
    print("Tweet:", tweet)
    print("Predicted Sentiment:", predict_sentiment(tweet))

Tweet: I love this new phone, battery life is good
Predicted Sentiment: Positive
Tweet: This laptop is very slow and performance is bad
Predicted Sentiment: Negative
